In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# =========================================================
# 1. Dataset
# =========================================================

df = pd.DataFrame({
    "age": [20, 25, 30, 35, 40, 28, 32, 45, 22, 38],
    "salary": [3000, 4500, 5000, 7000, 8000, 4200, 6000, 9000, 3500, 7500],
    "city": [
        "Safi", "Casa", "Rabat", "Safi", "Casa",
        "Rabat", "Safi", "Casa", "Rabat", "Safi"
    ],
    "education": [
        "Poor", "Average", "Good", "Good", "Average",
        "Poor", "Good", "Good", "Average", "Poor"
    ],
    "bought": [
        "No", "Yes", "Yes", "Yes", "Yes",
        "No", "Yes", "Yes", "No", "Yes"
    ]
})


# =========================================================
# 2. X / y
# =========================================================

X = df.drop("bought", axis=1)
y = df["bought"]


# =========================================================
# 3. Label Encoding de la target
# =========================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)


# =========================================================
# 4. Train / Test
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================================================
# 5. Identifier les types
# =========================================================

numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

city_features = ["city"]

education_features = ["education"]


# =========================================================
# 6. Numeric preprocessing
# =========================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# =========================================================
# 7. City → OneHotEncoder
# =========================================================

city_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# =========================================================
# 8. Education → OrdinalEncoder
# =========================================================

education_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["Poor", "Average", "Good"]]
    ))
])


# =========================================================
# 9. Combine preprocessing
# =========================================================

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("city", city_pipeline, city_features),
    ("education", education_pipeline, education_features)
])


# =========================================================
# 10. Fit uniquement sur Train
# =========================================================

X_train_processed = preprocessor.fit_transform(X_train)


# =========================================================
# 11. Transform Test
# =========================================================

X_test_processed = preprocessor.transform(X_test)


# =========================================================
# 12. Logistic Regression
# =========================================================

model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(
    X_train_processed,
    y_train
)

y_pred_lr = model_lr.predict(X_test_processed)


# =========================================================
# 13. Decision Tree
# =========================================================

model_dt = DecisionTreeClassifier(
    random_state=42
)

model_dt.fit(
    X_train_processed,
    y_train
)

y_pred_dt = model_dt.predict(X_test_processed)


# =========================================================
# 14. Random Forest
# =========================================================

model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_rf.fit(
    X_train_processed,
    y_train
)

y_pred_rf = model_rf.predict(X_test_processed)


# =========================================================
# 15. Evaluation function
# =========================================================

def evaluate_model(name, y_test, y_pred):

    print(f"\n===== {name} =====")

    print(
        "Accuracy :",
        accuracy_score(y_test, y_pred)
    )

    print(
        "Precision:",
        precision_score(y_test, y_pred)
    )

    print(
        "Recall   :",
        recall_score(y_test, y_pred)
    )

    print(
        "F1-score :",
        f1_score(y_test, y_pred)
    )

    print("\nConfusion Matrix:")

    print(
        confusion_matrix(y_test, y_pred)
    )

    print("\nClassification Report:")

    print(
        classification_report(y_test, y_pred)
    )


# =========================================================
# 16. Evaluate 3 models
# =========================================================

evaluate_model(
    "Logistic Regression",
    y_test,
    y_pred_lr
)

evaluate_model(
    "Decision Tree",
    y_test,
    y_pred_dt
)

evaluate_model(
    "Random Forest",
    y_test,
    y_pred_rf
)

C:\Users\moham\AppData\Local\Temp\ipykernel_31320\4105678969.py:89: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(



===== Logistic Regression =====
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0

Confusion Matrix:
[[1 0]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


===== Decision Tree =====
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0

Confusion Matrix:
[[1 0]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


===== Random Forest =====
Accuracy : 1.0
Precisio

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# =========================================================
# 1. Dataset
# =========================================================

df = pd.DataFrame({
    "age": [20, 25, 30, 35, 40, 28, 32, 45, 22, 38],
    "salary": [3000, 4500, 5000, 7000, 8000, 4200, 6000, 9000, 3500, 7500],
    "city": [
        "Safi", "Casa", "Rabat", "Safi", "Casa",
        "Rabat", "Safi", "Casa", "Rabat", "Safi"
    ],
    "education": [
        "Poor", "Average", "Good", "Good", "Average",
        "Poor", "Good", "Good", "Average", "Poor"
    ],
    "bought": [
        "No", "Yes", "Yes", "Yes", "Yes",
        "No", "Yes", "Yes", "No", "Yes"
    ]
})


# =========================================================
# 2. X / y
# =========================================================

X = df.drop("bought", axis=1)
y = df["bought"]


# =========================================================
# 3. Label Encoding de la target
# =========================================================

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


# =========================================================
# 4. Train / Test
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================================================
# 5. Identifier les types
# =========================================================

numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

city_features = ["city"]
education_features = ["education"]


# =========================================================
# 6. Numeric preprocessing
# =========================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# =========================================================
# 7. City → OneHotEncoder
# =========================================================

city_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# =========================================================
# 8. Education → OrdinalEncoder
# =========================================================

education_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["Poor", "Average", "Good"]]
    ))
])


# =========================================================
# 9. Combine preprocessing
# =========================================================

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("city", city_pipeline, city_features),
    ("education", education_pipeline, education_features)
])


# =========================================================
# 10. Train / Test preprocessing
# =========================================================

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


# =========================================================
# 11. Models
# =========================================================

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}


# =========================================================
# 12. Training + Prediction + Evaluation
# =========================================================

predictions = {}

for name, model in models.items():

    # Training
    model.fit(X_train_processed, y_train)

    # Prediction
    y_pred = model.predict(X_test_processed)

    # Save predictions
    predictions[name] = y_pred

    # Evaluation
    print(f"\n{'=' * 50}")
    print(name)
    print(f"{'=' * 50}")

    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1-score :", f1_score(y_test, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Logistic Regression
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0

Confusion Matrix:
[[1 0]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


Decision Tree
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0

Confusion Matrix:
[[1 0]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


Random Forest
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0

C:\Users\moham\AppData\Local\Temp\ipykernel_31320\3933820124.py:88: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(
